# Chapter 2: Growing the Book
### *Law of Large Numbers*

Arclight survived year one. The CEO landed Meridian Medical Group, priced the policy correctly, and the company ended the year with a small profit — or a small loss, depending on how the dice fell.

The board has approved an expansion. The sales team can now write policies for companies with the same profile as Meridian: mid-size healthcare providers, \$10M revenue, same cyber exposure.

**The board's question: how many policies do we need before our results become predictable?**

There is a deeper question lurking underneath it: *does writing more cyber policies actually make us safer, or are we just concentrating correlated risk?*

## The math

Let $X_i$ be the annual loss for policy $i$, with mean $\mu$ and variance $\sigma^2$. If all policies are **independent**, the portfolio loss ratio is:

$$\bar{L} = \frac{\sum_{i=1}^{n} X_i}{n \cdot P}$$

By the law of large numbers, $\bar{L} \to \mu / P$ as $n \to \infty$. The variance shrinks:

$$\text{Var}[\bar{L}] = \frac{\sigma^2}{n \cdot P^2} \quad \Rightarrow \quad \text{Std Dev}[\bar{L}] \propto \frac{1}{\sqrt{n}}$$

Write enough uncorrelated policies and your loss ratio becomes predictable — the foundation of the entire insurance industry.

**Systemic risk breaks this.** Suppose with probability $p$ each year a correlated event fires and hits fraction $r$ of your book simultaneously. The total loss is:

$$\text{Total loss} = \underbrace{\sum_i X_i^{\text{indep}}}_{\text{diversifies away}} + \underbrace{Z \cdot r \cdot n \cdot s}_{\text{does not diversify}}$$

where $Z \sim \text{Bernoulli}(p)$ and $s$ is the per-policy loss from the systemic event. As $n$ grows, the first term shrinks per-policy but the second term grows with $n$. The std dev of the loss ratio **floors out** at a non-zero value no matter how large the book gets.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# Same exposure profile as Chapter 1
THREATS = [
    ('Ransomware',                0.08, 0.0250),
    ('Data Breach',               0.04, 0.0200),
    ('Business Email Compromise', 0.10, 0.0040),
]

ANNUAL_REVENUE     = 10_000_000
EXPENSE_RATIO      = 0.35
PREMIUM_PER_POLICY = 55_000

FREQ = np.array([t[1] for t in THREATS])
SEV  = np.array([t[2] * ANNUAL_REVENUE for t in THREATS])
PURE = float((FREQ * SEV).sum())

print(f'Pure premium per policy:   ${PURE:,.0f}')
print(f'Gross premium per policy:  ${PURE / (1 - EXPENSE_RATIO):,.0f}')
print(f'Target premium per policy: ${PREMIUM_PER_POLICY:,.0f}  '
      f'(profit margin: {(PREMIUM_PER_POLICY - PURE / (1 - EXPENSE_RATIO)) / PREMIUM_PER_POLICY:.1%})')

Pure premium per policy:   $32,000
Gross premium per policy:  $49,231
Target premium per policy: $55,000  (profit margin: 10.5%)


In [2]:
def simulate_book(n_policies, n_sims, include_systemic,
                  systemic_prob, systemic_reach, rng):
    """Simulate n_sims years for a book of n_policies. Returns loss ratio array."""
    fires        = rng.random((n_sims, n_policies, len(SEV))) < FREQ
    total_claims = (fires * SEV).sum(axis=(1, 2))   # shape: (n_sims,)
    if include_systemic:
        events       = rng.random(n_sims) < systemic_prob
        n_hit        = max(1, round(n_policies * systemic_reach))
        total_claims = total_claims + events * (n_hit * SEV[0])  # ransomware-level per hit
    return total_claims / (n_policies * PREMIUM_PER_POLICY)


SWEEP_SIZES  = np.array([1, 5, 10, 25, 50, 100, 250, 500, 1000, 2000])
N_SWEEP_SIMS = 300

def compute_sweep(include_systemic, systemic_prob, systemic_reach, rng):
    """Std dev of loss ratio for each book size in SWEEP_SIZES."""
    stds_indep, stds_sys = [], []
    for n in SWEEP_SIZES:
        lr = simulate_book(n, N_SWEEP_SIMS, False, systemic_prob, systemic_reach, rng)
        stds_indep.append(lr.std())
        if include_systemic:
            lr_s = simulate_book(n, N_SWEEP_SIMS, True, systemic_prob, systemic_reach, rng)
            stds_sys.append(lr_s.std())
    return (np.array(stds_indep),
            np.array(stds_sys) if include_systemic else None)

## Your turn: grow the book

Choose how many policies to write. Run the simulation and watch the loss ratio distribution narrow.

Once you have a feel for the diversification benefit, toggle on **systemic risk** and set the probability and reach of a major correlated event — modeled as a ransomware campaign that simultaneously hits a fraction of your book (think WannaCry or NotPetya). Observe what happens to the sweep curve.

In [5]:
n_slider      = widgets.IntSlider(value=100, min=1, max=2000, step=10,
                                  description='Policies:',
                                  layout=widgets.Layout(width='500px'))
n_sims_slider = widgets.IntSlider(value=300, min=50, max=1000, step=50,
                                  description='Sim years:',
                                  layout=widgets.Layout(width='500px'))
sys_check     = widgets.Checkbox(value=False, description='Include systemic risk')
sys_prob_sl   = widgets.FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01,
                                    description='Sys. prob:',
                                    readout_format='.0%',
                                    layout=widgets.Layout(width='440px'))
sys_reach_sl  = widgets.FloatSlider(value=0.25, min=0.05, max=0.75, step=0.05,
                                    description='Sys. reach:',
                                    readout_format='.0%',
                                    layout=widgets.Layout(width='440px'))
run_button    = widgets.Button(description='Run simulation', button_style='primary')
output        = widgets.Output()

sys_box = widgets.VBox([sys_prob_sl, sys_reach_sl])

def toggle_sys(change):
    sys_box.layout.display = '' if change['new'] else 'none'

toggle_sys({'new': False})
sys_check.observe(toggle_sys, names='value')


def run_sim(_):
    n         = n_slider.value
    n_sims    = n_sims_slider.value
    inc_sys   = sys_check.value
    sys_prob  = sys_prob_sl.value
    sys_reach = sys_reach_sl.value
    rng       = np.random.default_rng()

    loss_ratios           = simulate_book(n, n_sims, inc_sys, sys_prob, sys_reach, rng)
    stds_indep, stds_sys  = compute_sweep(inc_sys, sys_prob, sys_reach, rng)

    mean_lr   = loss_ratios.mean()
    std_lr    = loss_ratios.std()
    p95_lr    = np.percentile(loss_ratios, 95)
    p99_lr    = np.percentile(loss_ratios, 99)
    loss_prob = (loss_ratios > 1.0).mean()

    with output:
        output.clear_output(wait=True)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

        ax1.hist(loss_ratios, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
        ax1.axvline(mean_lr, color='black', linestyle='--', linewidth=1.5,
                    label=f'Mean {mean_lr:.1%}')
        ax1.axvline(p95_lr,  color='crimson',    linestyle='--', linewidth=1.5,
                    label=f'95th pct {p95_lr:.1%}')
        ax1.axvline(1.0,     color='darkorange', linestyle=':',  linewidth=1.5,
                    label='Loss ratio = 100%')
        ax1.set_title(f'Annual loss ratio — {n} policies, {n_sims} simulated years')
        ax1.set_xlabel('Loss ratio')
        ax1.set_ylabel('Frequency')
        ax1.legend(fontsize=8)

        ax2.plot(SWEEP_SIZES, stds_indep * 100, 'o-', color='steelblue',
                 label='Independent risk only')
        if stds_sys is not None:
            ax2.plot(SWEEP_SIZES, stds_sys * 100, 's--', color='crimson',
                     label='+ Systemic risk')
        ax2.axvline(n, color='gray', linestyle=':', linewidth=1.5,
                    label=f'Your book (n={n})')
        ax2.set_xscale('log')
        ax2.set_title('Loss ratio volatility vs. book size')
        ax2.set_xlabel('Number of policies (log scale)')
        ax2.set_ylabel('Std dev of loss ratio (%)')
        ax2.legend(fontsize=8)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        lines = []
        lines.append(f'<p><b>Book: {n} policies | '
                     f'Premium: ${PREMIUM_PER_POLICY:,.0f}/policy | '
                     f'Annual book premium: ${n * PREMIUM_PER_POLICY:,.0f}</b></p>')
        lines.append(f'<p>Mean loss ratio: <b>{mean_lr:.1%}</b> &nbsp;|&nbsp; '
                     f'Std dev: <b>{std_lr:.1%}</b> &nbsp;|&nbsp; '
                     f'95th pct: <b>{p95_lr:.1%}</b> &nbsp;|&nbsp; '
                     f'99th pct: <b>{p99_lr:.1%}</b></p>')
        lines.append(f'<p>Loss years (loss ratio &gt; 100%): <b>{loss_prob:.1%}</b></p>')

        if not inc_sys:
            if n < 20:
                lines.append(f'<p style="color:darkorange"><b>Single-policy variance.</b> '
                             f'At {n} policies, one bad year dominates. '
                             f'Results are near-random.</p>')
            elif std_lr > 0.12:
                lines.append(f'<p style="color:darkorange"><b>Still volatile.</b> '
                             f'Std dev of {std_lr:.1%} is too high for reliable planning. '
                             f'Grow the book further.</p>')
            else:
                lines.append(f'<p style="color:seagreen"><b>Stable.</b> '
                             f'At {n} policies, std dev has fallen to {std_lr:.1%}. '
                             f'The LLN is doing its job — results are converging.</p>')
        else:
            floor = float(stds_sys[-1]) * 100 if stds_sys is not None else 0.0
            lines.append(f'<p style="color:crimson"><b>The floor.</b> '
                         f'Even at 2,000 policies, loss ratio volatility only falls to ~{floor:.1f}% '
                         f'with a {sys_prob:.0%}-per-year event hitting {sys_reach:.0%} of the book. '
                         f'Diversification cannot eliminate this tail.</p>')
            lines.append('<p>This is the structural challenge of cyber insurance. '
                         'Traditional lines (auto, home) have nearly independent risks. '
                         'Cyber does not: one piece of malware can fire across your entire book. '
                         'The only real mitigants are aggregate limits, reinsurance, and '
                         'limiting cyber concentration within your portfolio.</p>')

        lines.append('<hr><p><b>Key takeaway.</b> '
                     'Volume tames idiosyncratic risk: add independent policies and your loss ratio converges. '
                     'Systemic cyber risk is structurally different — correlated events hit the whole book '
                     'at once, and no amount of policy count eliminates that tail. '
                     'The LLN works until it does not.</p>')
        display(HTML(''.join(lines).replace('$', '&#36;')))


run_button.on_click(run_sim)
display(widgets.VBox([
    n_slider, n_sims_slider, sys_check, sys_box, run_button, output
]))

## Hints

<details><summary>Hint 1 — where does the variance come from?</summary>

With one policy, the annual loss is either \$0 (no incident), \$250K (ransomware), \$200K (breach), \$40K (BEC), or some combination. That is an enormous range relative to the \$55K premium.

</details>

<details><summary>Hint 2 — how many policies to stabilize?</summary>

Try n=50, 200, 500. Watch the histogram narrow. Roughly, variance halves every time you quadruple the book (because std dev ∝ 1/√n). By n=500 the std dev of the loss ratio should be well under 5% without systemic risk.

</details>

<details><summary>Hint 3 — the systemic floor calculation</summary>

With `sys_prob=0.05` (5%/year) and `sys_reach=0.25` (hits 25% of book), in the years the event fires:

```
n_hit     = 0.25 × n
sys_loss  = n_hit × $250,000
sys LR    = (0.25 × n × $250K) / (n × $55K) = 0.25 × $250K / $55K ≈ 113%
```

When the event fires, the portfolio loss ratio jumps by ~113 points regardless of book size. That is the floor.

</details>

<details><summary>Hint 4 — real-world analogue</summary>

WannaCry (2017) and NotPetya (2017) each infected hundreds of thousands of systems across multiple industries worldwide in a matter of days. Any insurer with significant cyber concentration took correlated losses across their entire book simultaneously.

</details>

## What's next

**Chapter 3 — Fitting the Curve (Loss Distributions).** You have been pricing with a fixed severity: ransomware always costs exactly \$250K. That is a useful simplification but wrong in practice — actual cyber losses range from a few thousand dollars to tens of millions, and the distribution is heavily right-skewed.

Arclight is now two years in with a book of 800 policies. The CEO wants to launch a *Cyber Premier* product with coverage limits up to \$5M for larger clients. To price those high limits correctly, you need to understand the **shape** of the loss distribution, not just its mean. A thin-tailed assumption will dramatically underprice the upper layers — and that is how insurers go insolvent.